In [5]:
import sys
from pathlib import Path

DEFS_DIR = (Path.cwd() / ".." / "defs").resolve()
MODELS_DIR = (Path.cwd() / ".." / "models").resolve()

print("DEFS_DIR:", DEFS_DIR, "exists:", DEFS_DIR.exists())
print("MODELS_DIR:", MODELS_DIR, "exists:", MODELS_DIR.exists())

for d in [DEFS_DIR, MODELS_DIR]:
    if d.exists() and str(d) not in sys.path:
        sys.path.insert(0, str(d))

print("sys.path[0:5] =", sys.path[:5])

DEFS_DIR: /Users/sammuelaldrichkarya/Documents/Projects/UTAT/FINCH-Science_DLUnmixing/defs exists: True
MODELS_DIR: /Users/sammuelaldrichkarya/Documents/Projects/UTAT/FINCH-Science_DLUnmixing/models exists: True
sys.path[0:5] = ['/Users/sammuelaldrichkarya/Documents/Projects/UTAT/FINCH-Science_DLUnmixing/models', '/Users/sammuelaldrichkarya/Documents/Projects/UTAT/FINCH-Science_DLUnmixing/defs', '/Users/sammuelaldrichkarya/.pyenv/versions/3.12.3/lib/python312.zip', '/Users/sammuelaldrichkarya/.pyenv/versions/3.12.3/lib/python3.12', '/Users/sammuelaldrichkarya/.pyenv/versions/3.12.3/lib/python3.12/lib-dynload']


In [6]:
from pathlib import Path

SEED = 42
SIZES = ["m", "l", "xl"]

CNN_RUNS_ROOT = Path("../runs")
FNO_RUNS_ROOT = Path("../runs")

print("CNN_RUNS_ROOT:", CNN_RUNS_ROOT.resolve())
print("FNO_RUNS_ROOT:", FNO_RUNS_ROOT.resolve())

CNN_RUNS_ROOT: /Users/sammuelaldrichkarya/Documents/Projects/UTAT/FINCH-Science_DLUnmixing/runs
FNO_RUNS_ROOT: /Users/sammuelaldrichkarya/Documents/Projects/UTAT/FINCH-Science_DLUnmixing/runs


In [7]:
def find_latest_run_dir(runs_root: Path, prefix: str) -> Path | None:
    candidates = [p for p in runs_root.iterdir() if p.is_dir() and p.name.startswith(prefix)]
    if not candidates:
        return None
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

In [8]:
cnn_run_dirs = {}
for s in SIZES:
    preferred = f"cnn_{s}_seed{SEED}"
    fallback  = f"cnn_{s}"
    d = find_latest_run_dir(CNN_RUNS_ROOT, preferred) or find_latest_run_dir(CNN_RUNS_ROOT, fallback)
    cnn_run_dirs[s] = d

fno_run_dirs = {}
for s in SIZES:
    preferred = f"fno_{s}_seed{SEED}"
    fallback  = f"fno_{s}"
    d = find_latest_run_dir(FNO_RUNS_ROOT, preferred) or find_latest_run_dir(FNO_RUNS_ROOT, fallback)
    fno_run_dirs[s] = d

cnn_run_dirs, fno_run_dirs


({'m': PosixPath('../runs/cnn_m_seed42_20260122_163308_seed42'),
  'l': PosixPath('../runs/cnn_l_seed42_20260122_233252_seed42'),
  'xl': PosixPath('../runs/cnn_xl_seed42_20260123_112702_seed42')},
 {'m': PosixPath('../runs/fno_m_seed42'),
  'l': PosixPath('../runs/fno_l_seed42'),
  'xl': PosixPath('../runs/fno_xl_seed42')})

In [ ]:
import pandas as pd
import torch
import json

from auxiliary import get_n_params
from cnn_unmixing import CNNUnmixing1D
from fno_unmixing import FNO1DUnmixing

def load_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


def cnn_init_kwargs(summary: dict) -> dict:
    # CNN args live here
    return summary["config"]["model"]


def fno_init_kwargs(summary: dict) -> dict:
    kwargs = dict(summary["model"])  # modes/width/num_layers/dropout/pool

    target_cols = summary.get("data", {}).get(
        "target_cols",
        ["gv_fraction", "npv_fraction", "soil_fraction"]
    )
    kwargs["num_endmembers"] = len(target_cols)  # K

    return kwargs


rows = []

# ---- CNN ----
for size, run_dir in cnn_run_dirs.items():
    if run_dir is None:
        continue

    summary = load_json(run_dir / "summary.json")
    kwargs = cnn_init_kwargs(summary)

    model = CNNUnmixing1D(**kwargs)
    rows.append({
        "model": "CNN",
        "size": size.upper(),
        "n_params": int(get_n_params(model)),
    })

# ---- FNO ----
for size, run_dir in fno_run_dirs.items():
    if run_dir is None:
        continue

    summary = load_json(run_dir / "summary.json")
    kwargs = fno_init_kwargs(summary)

    model = FNO1DUnmixing(**kwargs)
    rows.append({
        "model": "FNO",
        "size": size.upper(),
        "n_params": int(get_n_params(model)),
    })

params_df = pd.DataFrame(rows).sort_values(["model", "size"]).reset_index(drop=True)
# params_df.to_csv("model_params_seed42.csv", index=False)
params_df

,model,size,n_params
0,CNN,L,6146019
1,CNN,M,1527363
2,CNN,XL,25804675
3,FNO,L,14304963
4,FNO,M,4260995
5,FNO,XL,42273283
